# Query Translation

**Query Translation** is the process of rewriting, expanding, or transforming a user's query into a better form before retrieval. The goal is to improve the quality of retrieved documents by making the search query more relevant and expressive.

Instead of directly searching with the original user query, the system first generates a more effective query (or multiple queries) and then performs retrieval.

---

# Multi-Query Retrieval

**Multi-Query Retrieval** is a query translation technique where an LLM generates multiple variations of the same user query. Each variation is used to retrieve relevant documents, and the retrieved results are combined before being passed to the LLM for answer generation.

### Why use Multi-Query Retrieval?

- Improves retrieval recall by searching with multiple query variations.
- Handles different wording, synonyms, and expressions.
- Increases the chance of retrieving the most relevant context.
- Produces more robust and context-aware answers.

### Workflow

```text
User Question
      │
      ▼
Generate Multiple Query Variations
      │
 ┌────┼────┐
 │    │    │
 ▼    ▼    ▼
Query1 Query2 Query3
 │      │      │
 ▼      ▼      ▼
Retriever Retriever Retriever
      │
      ▼
Merge Retrieved Documents
      │
      ▼
LLM
      │
      ▼
Final Answer
```

### Key Difference

- **Query Translation:** The general process of transforming a user's query before retrieval.
- **Multi-Query Retrieval:** A specific query translation technique that generates multiple query variations to improve retrieval performance.


## Implementing Multi-Query Retrieval

### Changes Made

- Multi-Query Retrieval implement gareko jasko purpose retrieval quality improve garne ho.
- OpenAI ko satta local **ChatOllama** use gareko.
- LLM bata user ko question ko multiple variations generate gareko.
- Generate bhayeka pratyek query ko lagi relevant documents retrieve gareko.
- Retrieve bhayeka duplicate documents remove garera unique documents matra rakheko.
- Retrieved context ra original user question combine garera final prompt create gareko.
- Local **Llama 3** model use garera final answer generate gareko.
- OpenAI API key ko requirement hataera pura workflow local ra free banayeko.


In [1]:
#### INDEXING ####

import bs4

# Website bata document load garna
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

# Website ko content load gareko
blog_docs = loader.load()


# Document lai token-based chunks ma split garna
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

# Document lai sano-sano chunks ma divide gareko
splits = text_splitter.split_documents(blog_docs)


# HuggingFace bata free embedding model use gareko
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")


# Chroma vector database ma document ko embeddings store gareko
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
)


# Vector database bata semantic search garna retriever create gareko
retriever = vectorstore.as_retriever()

C:\Users\Acer\AppData\Local\Temp\ipykernel_2396\1570852886.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# Multi-Query prompt create gareko
from langchain_core.prompts import ChatPromptTemplate

template = """
You are an AI language model assistant.

Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database.

By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of distance-based similarity search.

Provide these alternative questions separated by new lines.

Original question: {question}
"""

# Multi-query generation ko lagi prompt template create gareko
prompt_perspectives = ChatPromptTemplate.from_template(template)


# Output parser import gareko
from langchain_core.output_parsers import StrOutputParser

# Local Ollama LLM use gareko (OpenAI ko satta)
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Prompt → LLM → Output Parser → List of Queries banne chain create gareko
generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    # LLM le generate gareko queries lai newline ko basis ma list ma split gareko
    | (lambda x: x.split("\n"))
)

In [4]:
from langchain_core.load import dumps, loads


# Duplicate documents remove garera unique documents return garne function
def get_unique_union(documents: list[list]):
    """Retrieved documents bata unique documents matra return garne"""

    # List of lists lai flatten garera Document lai string ma convert gareko
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]

    # Duplicate documents remove gareko
    unique_docs = list(set(flattened_docs))

    # String lai feri Document object ma convert gareko
    return [loads(doc) for doc in unique_docs]


# User ko question
question = "What is task decomposition for LLM agents?"


# Multi-query generate garne → Pratyek query ko lagi retrieval garne →
# Duplicate documents remove garera unique documents return garne chain
retrieval_chain = generate_queries | retriever.map() | get_unique_union


# Retrieval chain execute gareko
docs = retrieval_chain.invoke({"question": question})


# Retrieve bhayeko unique documents ko number print garne
print(len(docs))

16


C:\Users\Acer\AppData\Local\Temp\ipykernel_2396\3729604343.py:15: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
C:\Users\Acer\AppData\Local\Temp\ipykernel_2396\3729604343.py:15: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


In [5]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# RAG prompt create gareko
template = """
Answer the following question based only on the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)


# Local Ollama LLM use gareko (OpenAI ko satta)
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Final RAG chain create gareko
final_rag_chain = (
    {
        # Multi-query retrieval bata aayeko unique documents context ma pathaune
        "context": retrieval_chain,
        # User ko original question prompt ma pathaune
        "question": itemgetter("question"),
    }
    # Context ra question prompt template ma inject garne
    | prompt
    # Prompt lai local Ollama LLM ma pathaune
    | llm
    # LLM ko response lai plain text ma convert garne
    | StrOutputParser()
)


# Final RAG chain execute gareko
response = final_rag_chain.invoke({"question": question})

# Generate bhayeko final answer print garne
print(response)

C:\Users\Acer\AppData\Local\Temp\ipykernel_2396\3729604343.py:15: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


Based on the provided context, task decomposition for LLM (Large Language Model) agents involves breaking down a problem into multiple thought steps or subgoals. This can be done through:

1. Simple prompting: Using prompts like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?" to guide the LLM.
2. Task-specific instructions: Providing task-specific instructions, such as "Write a story outline," to help the LLM decompose the problem.
3. Human inputs: Incorporating human input into the decomposition process.

This approach allows the LLM agent to generate multiple thoughts per step, creating a tree structure that can be searched using algorithms like BFS (breadth-first search) or DFS (depth-first search). The search process is evaluated by a classifier or majority vote.
